# 01 — Data: Build Manifests\n
\n
This notebook will download/prepare datasets and write unified manifests to `data/manifests/`.\n
\n
Datasets:\n
- Underwater detection: Fish Occlusion Dataset (FOD) — https://doi.org/10.57760/sciencedb.27669\n
- Taxonomy classification: FishNet — https://fishnet-2023.github.io/\n
\n
Manifests will be JSONL with one record per image (or per crop for taxonomy).

In [1]:
from pathlib import Path
import sys

ROOT = Path('..').resolve()
DATA_DIR = (ROOT / 'data').resolve()
MANIFEST_DIR = DATA_DIR / 'manifests'
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str((ROOT / 'src').resolve()))

print('ROOT:', ROOT)
print('MANIFEST_DIR:', MANIFEST_DIR)


ROOT: /Users/sushantshah/Desktop/Fish Codes
MANIFEST_DIR: /Users/sushantshah/Desktop/Fish Codes/data/manifests


## FOD (Fish Occlusion Dataset) → fish-only box manifests\n
\n
Download FOD from the Science Data Bank (DOI) and place files under:\n
- `data/raw/fod/`\n
\n
Expected structure (example):\n
- `data/raw/fod/images/` (all images)\n
- `data/raw/fod/annotations/instances.json` (COCO instances)\n
\n
Source: https://doi.org/10.57760/sciencedb.27669

In [2]:
from fish_ai.data.fod_manifest import (
    SplitConfig,
    build_fod_detection_rows,
    split_rows,
    write_split_manifests,
)

# Your FOD download has the actual payload under `data/raw/FOD/data/...`
fod_root = DATA_DIR / 'raw' / 'FOD'
fod_payload = fod_root / 'data'

fod_images = fod_payload / 'images'  # may contain train/ and test/
fod_ann_train = fod_payload / 'annotations' / 'instances_train.json'
fod_ann_test = fod_payload / 'annotations' / 'instances_test.json'

print('fod_images exists:', fod_images.exists())
print('fod_ann_train exists:', fod_ann_train.exists())
print('fod_ann_test exists:', fod_ann_test.exists())
print('fod_images/train exists:', (fod_images / 'train').exists())
print('fod_images/test exists:', (fod_images / 'test').exists())



fod_images exists: True
fod_ann_train exists: True
fod_ann_test exists: True
fod_images/train exists: True
fod_images/test exists: True


In [3]:
# Optional: inspect what's inside annotations/
from pathlib import Path

ann_dir = fod_payload / 'annotations'
print('annotation dir:', ann_dir)
if ann_dir.exists():
    for p in sorted(ann_dir.glob('*')):
        print(p.name, 'bytes=', p.stat().st_size)


annotation dir: /Users/sushantshah/Desktop/Fish Codes/data/raw/FOD/data/annotations
instances_test.json bytes= 150547589
instances_train.json bytes= 484767286


In [4]:
# Build train/test rows from the provided COCO files
train_rows = build_fod_detection_rows(fod_ann_train, fod_images)
test_rows = build_fod_detection_rows(fod_ann_test, fod_images)

print('train images:', len(train_rows))
print('test images:', len(test_rows))

# Create val split from train only; keep test as provided
train_val = split_rows(train_rows, SplitConfig(train_frac=0.9, val_frac=0.1, test_frac=0.0, seed=42))

splits = {
    'train': train_val['train'],
    'val': train_val['val'],
    'test': test_rows,
}

out_paths = write_split_manifests(splits, MANIFEST_DIR, prefix='fod_detection')
out_paths


train images: 10940
test images: 3436


{'train': PosixPath('/Users/sushantshah/Desktop/Fish Codes/data/manifests/fod_detection_train.jsonl'),
 'val': PosixPath('/Users/sushantshah/Desktop/Fish Codes/data/manifests/fod_detection_val.jsonl'),
 'test': PosixPath('/Users/sushantshah/Desktop/Fish Codes/data/manifests/fod_detection_test.jsonl')}

## Next steps (to implement)\n
\n
1. Download FOD COCO annotations + images to `data/raw/fod/`.\n
2. Convert COCO instances to fish-only **boxes** and write `fod_detection_{train,val,test}.jsonl`.\n
3. Download FishNet taxonomy/classification split(s) to `data/raw/fishnet/` (manual step; link is on the project page).\n
4. Filter to top-100 species and write `fishnet_taxonomy_{train,val,test}.jsonl` with family/genus/species fields.\n
\n
NOTE: FishNet manifest generation is the next section to implement.

## FishNet → taxonomy manifests (top-100 species)

FishNet images are downloaded separately from the project page (Google Drive link).
Place them under `data/raw/fishnet/images/` (either flat or nested).

FishNet annotation CSVs are in the FishNet GitHub repo under `anns/`.
Place them under `data/raw/fishnet/anns/`:
- `train.csv`
- `test.csv`

Sources:
- https://fishnet-2023.github.io/
- https://github.com/faixan-khan/FishNet

In [5]:
from fish_ai.data.fishnet_manifest import FishNetLayout, write_fishnet_taxonomy_manifests

fishnet_root = DATA_DIR / 'raw' / 'fishnet'
fishnet_images = fishnet_root / 'images'
fishnet_anns = fishnet_root / 'anns'
train_csv = fishnet_anns / 'train.csv'
test_csv = fishnet_anns / 'test.csv'

print('fishnet_images exists:', fishnet_images.exists())
print('train_csv exists:', train_csv.exists())
print('test_csv exists:', test_csv.exists())

fishnet_images exists: True
train_csv exists: True
test_csv exists: True


In [6]:
# Writes: fishnet_taxonomy_{train,val,test}.jsonl (top-100 species)
layout = FishNetLayout(train_csv=train_csv, test_csv=test_csv, images_root=fishnet_images)
paths = write_fishnet_taxonomy_manifests(layout, MANIFEST_DIR, top_n_species=100, val_frac_from_train=0.1, seed=42)
paths

{'train': PosixPath('/Users/sushantshah/Desktop/Fish Codes/data/manifests/fishnet_taxonomy_train.jsonl'),
 'val': PosixPath('/Users/sushantshah/Desktop/Fish Codes/data/manifests/fishnet_taxonomy_val.jsonl'),
 'test': PosixPath('/Users/sushantshah/Desktop/Fish Codes/data/manifests/fishnet_taxonomy_test.jsonl')}